<a href="https://colab.research.google.com/github/khizertouseef76-hue/Flyrank_ML_Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/khizertouseef76-hue/Flyrank_Starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [3]:
import os, sys, subprocess

# 1. Detect environment
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Set your repository name and URL
    REPO_DIR = "Flyrank_ML_Internship"
    REPO_URL = f"https://github.com/khizertouseef76-hue/{REPO_DIR}.git"

    # Clone repo if not already present in Colab
    if not os.path.isdir(f"/content/{REPO_DIR}"):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, f"/content/{REPO_DIR}"], check=True)

    # Move working directory to repo root
    os.chdir(f"/content/{REPO_DIR}")

# 2. For local running: Navigate up if currently in work/notebooks
while not os.path.exists("data/raw/content_refresh_anonymized.csv") and os.getcwd() != "/":
    os.chdir("..")

print("Current Working Directory:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "Starter CSV not found!"
print("Data file located successfully!")

Current Working Directory: /content/Flyrank_ML_Internship
Data file located successfully!


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

# Load starter CSV
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Verify total count and target distribution
total_count = len(df)
declining_count = (df["trend_direction"].str.lower() == "down").sum()

print(f"Total rows (pages): {total_count:,}")
print(f"Declining class (Positive class): {declining_count:,} ({declining_count/total_count:.1%})")

Total rows (pages): 30,000
Declining class (Positive class): 16,262 (54.2%)


ML Task Framing: Probabilistic Binary Classification & Learning to Rank.  We frame Lane 2 (Content Refresh Opportunity Scoring) as a binary classification problem—predicting the probability $P(\text{is\_declining} = 1)$ for every page. These predicted probabilities are then scaled into a ranked score (0–100). Because human content reviewers operate under limited capacity, ranking candidates by model confidence ensures that editorial effort goes to the highest-risk pages first.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target Definition: is_declining_label = (trend_direction == "down").  
Label Source: In this starter dataset, this acts as an observed proxy label derived from traffic trends over the observation window.  
Production Reality: In a full production warehouse setup, the ideal target is a forward-looking outcome (e.g., whether a page's traffic drops by $>20\%$ over the next 30 days) to prevent data leakage. For this initial milestone, the 90-day trend proxy establishes a clean baseline.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Create the binary target variable explicitly
df["target"] = (df["trend_direction"].str.lower() == "down").astype(int)

# Inspect target balance
print("Target Value Counts (1 = Declining, 0 = Healthy/Up/Flat):")
print(df["target"].value_counts(normalize=True).round(3))

Target Value Counts (1 = Declining, 0 = Healthy/Up/Flat):
target
1    0.542
0    0.458
Name: proportion, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Primary Metric: Precision@50 (and Precision@K).  Why this metric: Accuracy is misleading because content teams do not review thousands of pages at once. Content editors review a fixed queue of candidates (e.g., top 50 pages per batch). Precision@50 measures: "Of the top 50 pages the model recommends for refresh, how many are actually declining?"  Target Benchmark: The simple baseline rule achieves Precision@50 of $\approx 0.24$ (12/50 correct). A good ML model targets Precision@50 $\ge 0.70$ (35/50 correct)

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Demonstrate Precision@50 concept on a dummy baseline vs target
df_sorted_by_impressions = df.sort_values(by="impressions_90d", ascending=False)
top_50_baseline = df_sorted_by_impressions.head(50)
baseline_p50 = top_50_baseline["target"].mean()

print(f"Volume-only Baseline Precision@50: {baseline_p50:.3f} ({int(baseline_p50*50)}/50 correct)")
print("Target Machine Learning Model Precision@50: >= 0.700 (35/50 correct)")

Volume-only Baseline Precision@50: 0.420 (21/50 correct)
Target Machine Learning Model Precision@50: >= 0.700 (35/50 correct)


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Unit of Analysis: One row = One unique content item / page (content_id) observed over a 90-day performance window.  The grain is page-level. Each row encapsulates search signals (impressions, CTR, average position), engagement signals (sessions, scroll rate), and content metadata (word count, content age).  

In [7]:
# This cell is for CODE (numbers, a query, a check).# Show unit of analysis (Grain check)
sample_cols = [
    "content_id",
    "impressions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "trend_direction"
]

print(f"Dataframe Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print("\nFirst 3 rows representing individual web pages:")
df[sample_cols].head(3)
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Dataframe Shape: 30,000 rows x 45 columns

First 3 rows representing individual web pages:


,content_id,impressions_90d,ctr,avg_position,content_age_days,trend_direction
0,content_304f48230142,3803,0.76,10.6,187,down
1,content_a1fb4e703a9e,15320,0.05,20.3,445,down
2,content_9aa793d4d895,12581,0.09,36.5,141,down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Non-linear Multi-Factor Interactions: Content decay is rarely caused by a single factor. An old page (content_age_days > 300) might perform well if it holds Position 1, while a newer page with dropping CTR and declining engagement requires immediate attention.  Fragility of Hardcoded Cutoffs: A rule like IF age > 180 AND impressions > 500 misses declining pages at 490 impressions or 175 days. Decision trees and random forests learn soft, continuous threshold boundaries across multiple features simultaneously.  

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Code proof: Show that single features alone fail to separate declining pages cleanly
print("Median values comparison between Declining (1) and Non-Declining (0) pages:")
features_to_check = ["impressions_90d", "content_age_days", "avg_position", "ctr"]
print(df.groupby("target")[features_to_check].median().round(2))

Median values comparison between Declining (1) and Non-Declining (0) pages:
        impressions_90d  content_age_days  avg_position   ctr
target                                                       
0                 472.0             287.0         10.05  0.04
1                 961.0             216.0         11.30  0.08


## Self-check

Before you submit, confirm each line honestly:

- [ yes] Every section above is filled — markdown thinking AND the code that backs it
- [yes ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ yes] No client names, URLs, or private queries anywhere
- [yes ] My claims use careful words: observed, measured, directional, decision-support
- [yes ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.